In [1]:
import pandas as pd
import duckdb

In [2]:
urls = [
    "https://storage.googleapis.com/gresearch/goemotions/data/full_dataset/goemotions_1.csv",
    "https://storage.googleapis.com/gresearch/goemotions/data/full_dataset/goemotions_2.csv",
    "https://storage.googleapis.com/gresearch/goemotions/data/full_dataset/goemotions_3.csv",
]
raw = pd.concat([pd.read_csv(u) for u in urls], ignore_index=True)
print(raw.shape)
raw.head()

(211225, 37)


,text,id,author,subreddit,link_id,parent_id,created_utc,rater_id,example_very_unclear,admiration,...,love,nervousness,optimism,pride,realization,relief,remorse,sadness,surprise,neutral
0,That game hurt.,eew5j0j,Brdd9,nrl,t3_ajis4z,t1_eew18eq,1.548381e+09,1,False,0,...,0,0,0,0,0,0,0,1,0,0
1,>sexuality shouldn’t be a grouping category I...,eemcysk,TheGreen888,unpopularopinion,t3_ai4q37,t3_ai4q37,1.548084e+09,37,True,0,...,0,0,0,0,0,0,0,0,0,0
2,"You do right, if you don't care then fuck 'em!",ed2mah1,Labalool,confessions,t3_abru74,t1_ed2m7g7,1.546428e+09,37,False,0,...,0,0,0,0,0,0,0,0,0,1
3,Man I love reddit.,eeibobj,MrsRobertshaw,facepalm,t3_ahulml,t3_ahulml,1.547965e+09,18,False,0,...,1,0,0,0,0,0,0,0,0,0
4,"[NAME] was nowhere near them, he was by the Fa...",eda6yn6,American_Fascist713,starwarsspeculation,t3_ackt2f,t1_eda65q2,1.546669e+09,2,False,0,...,0,0,0,0,0,0,0,0,0,1


In [6]:
query = r"""
WITH rater_rows AS (
  SELECT
    id, rater_id, text, example_very_unclear,
    (amusement + excitement + joy + love + desire + optimism + caring +
     pride + admiration + gratitude + relief + approval) AS n_pos,
    (fear + nervousness + remorse + embarrassment + disappointment +
     sadness + grief + disgust + anger + annoyance + disapproval) AS n_neg
  FROM raw
),
per_comment AS (
  SELECT
    id,
    COUNT(DISTINCT rater_id) AS n_raters,
    MAX(len(regexp_split_to_array(trim(text), '\s+'))) AS n_words,
    MAX(CASE WHEN example_very_unclear THEN 1 ELSE 0 END) AS has_unclear,
    MAX(CASE WHEN n_pos > 0 AND n_neg = 0 THEN 1 ELSE 0 END) AS has_pos_only,
    MAX(CASE WHEN n_neg > 0 AND n_pos = 0 THEN 1 ELSE 0 END) AS has_neg_only
  FROM rater_rows
  GROUP BY id
),
three_rater_comments AS (
  SELECT
    *,
    CASE WHEN has_pos_only = 1 AND has_neg_only = 1 THEN 1 ELSE 0 END AS conflict,
    CASE
      WHEN n_words <= 7 THEN '1-7 words'
      WHEN n_words <= 12 THEN '8-12 words'
      WHEN n_words <= 18 THEN '13-18 words'
      ELSE '19+ words'
    END AS length_bin,
    CASE
      WHEN n_words <= 7 THEN 1
      WHEN n_words <= 12 THEN 2
      WHEN n_words <= 18 THEN 3
      ELSE 4
    END AS length_bin_order
  FROM per_comment
  WHERE n_raters = 3
)
SELECT
  length_bin,
  COUNT(*) AS n_comments,
  SUM(conflict) AS n_conflict,
  ROUND(100.0 * AVG(conflict), 1) AS conflict_pct,
  SUM(has_unclear) AS n_with_unclear
FROM three_rater_comments
GROUP BY length_bin_order, length_bin
ORDER BY length_bin_order;
"""
result = duckdb.sql(query).df()
result

,length_bin,n_comments,n_conflict,conflict_pct,n_with_unclear
0,1-7 words,10364,295.0,2.8,372.0
1,8-12 words,9202,460.0,5.0,169.0
2,13-18 words,9513,538.0,5.7,163.0
3,19+ words,8256,527.0,6.4,141.0
